# Loading libraries

In [ ]:
import pandas as pd
import numpy as np

# Loading datasets

In [ ]:
pop_data_missense_annovar = pd.read_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/datasets_final/pop_data_missense_annovar.csv",
                                        sep="\t")


In [ ]:
final_clinpgx_processed_dataset_annovar = pd.read_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/datasets_final/final_clinpgx_processed_dataset_Annovar.hg38_multianno.txt",
                                                  sep="\t")


final_clinpgx_processed_dataset_annovar = final_clinpgx_processed_dataset_annovar.drop(columns=final_clinpgx_processed_dataset_annovar.iloc[:,174:202].columns, axis=1)
final_clinpgx_processed_dataset_annovar = final_clinpgx_processed_dataset_annovar.drop(columns="Otherinfo")

final_clinpgx_processed_dataset_annovar = final_clinpgx_processed_dataset_annovar.drop_duplicates()

final_clinpgx_processed_dataset_annovar['User_input'] = (
    final_clinpgx_processed_dataset_annovar['Chr'].astype(str) + ":" +
    final_clinpgx_processed_dataset_annovar['Start'].astype(str) + ":" +
    final_clinpgx_processed_dataset_annovar['Ref'].astype(str) + ":" +
    final_clinpgx_processed_dataset_annovar['Alt'].astype(str) + ":" +
    final_clinpgx_processed_dataset_annovar['Gene.refGene'].astype(str)
)

final_clinpgx_processed_dataset_annovar.replace(".", np.nan, inplace=True)
final_clinpgx_processed_dataset_annovar.replace("NA", np.nan, inplace=True)

final_clinpgx_processed_dataset_annovar.drop(columns=["Chr", "Start", "End", "Ref", "Alt"], inplace=True)
final_clinpgx_processed_dataset_annovar = final_clinpgx_processed_dataset_annovar[final_clinpgx_processed_dataset_annovar.columns[-1:].tolist() + final_clinpgx_processed_dataset_annovar.columns[:-1].tolist()]

# Data processing

In [ ]:
pop_data_missense_annovar_cols = pop_data_missense_annovar.columns.to_list()
final_clinpgx_processed_dataset_annovar_cols = final_clinpgx_processed_dataset_annovar.columns.to_list()

In [ ]:
common_cols = pop_data_missense_annovar.columns.intersection(
    final_clinpgx_processed_dataset_annovar.columns
    ).to_list()

In [ ]:
def subset_for_common(df):
  df = df[common_cols]
  return df


pop_data_missense_annovar = subset_for_common(pop_data_missense_annovar)
final_clinpgx_processed_dataset_annovar = subset_for_common(final_clinpgx_processed_dataset_annovar)

In [ ]:
annovar_dataset = pd.concat([pop_data_missense_annovar, final_clinpgx_processed_dataset_annovar], axis=0).reset_index(drop=True)

In [ ]:
annovar_dataset.drop_duplicates(inplace=True)

In [ ]:
duplicates = annovar_dataset['User_input'].value_counts()
duplicates = duplicates[duplicates > 1]
duplicates = duplicates.index.tolist()


In [ ]:
def deduplicate(df):
  df.update(df.groupby('User_input').ffill())
  df_deduped = df.drop_duplicates(subset=['User_input'], keep='last')
  return df_deduped

annovar_dataset_deduplicate = deduplicate(annovar_dataset)

In [ ]:
annovar_dataset_deduplicate['InterVar_automated'] = annovar_dataset_deduplicate['InterVar_automated'].map({'Likely benign': 'Likely_benign',
                                                                                                           'Likely pathogenic': 'Likely_pathogenic',
                                                                                                           'Uncertain significance': 'Uncertain_significance'
                                                                                                           }
                                                                                                          )

In [ ]:
acmg_map = {'Benign': 0, 'Likely_benign': 1, 'Uncertain_significance': 2, 'Likely_pathogenic': 3, 'Pathogenic': 4}
polyphen_map = {'D': 1.0, 'P': 0.5, 'B': 0.0}


annovar_dataset_deduplicate['Polyphen2_HVAR_pred'] = annovar_dataset_deduplicate['Polyphen2_HVAR_pred'].map(polyphen_map)
annovar_dataset_deduplicate['Polyphen2_HDIV_pred'] = annovar_dataset_deduplicate['Polyphen2_HDIV_pred'].map(polyphen_map)
annovar_dataset_deduplicate['InterVar_automated'] = annovar_dataset_deduplicate['InterVar_automated'].map(acmg_map)

In [ ]:
annovar_dataset_deduplicate.to_csv("/Users/ricardo/Desktop/MECD/2ndYear/Thesis/datasets_final/annovar_dataset_final.csv",
                                   index=False,
                                   sep="\t")

In [ ]:
annovar_dataset_deduplicate['Polyphen2_HDIV_pred'].unique()